# Anthropic API: Claude Features and OpenAI Comparisons

Use Python to explore Claude's native Messages API and understand how it differs from OpenAI's Responses API.

[Claude Console and API keys](https://platform.claude.com/) · [Messages guide](https://platform.claude.com/docs/en/build-with-claude/working-with-messages) · [Model pricing](https://platform.claude.com/docs/en/about-claude/pricing)

Both providers support the same broad workflow: instructions + input + model → generated output. The native SDK methods, message envelopes, tools and state handling differ. A gateway or compatibility layer may hide some differences, but does not guarantee feature parity.

In [ ]:
# Pylint treats notebook cells as a script and pip magics as pass statements.
# pylint: disable=missing-module-docstring,wrong-import-position,wrong-import-order,unnecessary-pass
%pip install anthropic openai pandas pydantic --upgrade --quiet

In [1]:
# pylint: disable=missing-module-docstring,wrong-import-position,invalid-name
MODEL = "claude-haiku-5-5"
MAX_TOKENS = 4096
RUN_OPENAI_COMPARISON = True
RUN_VISION = True
RUN_PDF = True
RUN_CACHING = True
RUN_SEARCH = True
RUN_CODE_EXECUTION = True
RUN_BATCH = True
OPENAI_MODEL = "gpt-6-luna"
SERVER_TOOL_MODEL = "claude-sonnet-5-5"
IMAGE_URL = (
    "https://raw.githubusercontent.com/BrightPool/udemy-prompt-engineering-course/"
    "main/images/function-calling-steps.png"
)
PDF_PATH = ""
BATCH_ID = ""

## Connect with the Anthropic SDK and inspect models

Create an [Anthropic API key](https://platform.claude.com/) and enter it in the hidden prompt. It is separate from an OpenAI key. The official [Anthropic Python SDK](https://platform.claude.com/docs/en/cli-sdks-libraries/sdks/python) handles authentication, requests and typed responses through `Anthropic()`.

Discover models instead of assuming that a model name from a video remains available. Check the relevant guide for tool and modality support.

The defaults use Claude Haiku 5.5 for fast examples, Claude Sonnet 5.5 for hosted Python, and GPT-6 Luna for the OpenAI comparison. Claude calls use low effort and allow room for thinking tokens; the short OpenAI classification uses no reasoning. [Claude models](https://platform.claude.com/docs/en/models/overview) · [GPT-6 Luna](https://developers.openai.com/api/docs/models/gpt-6-luna)

In [ ]:
import base64
import json
import os
import time
from getpass import getpass
from pathlib import Path

from anthropic import Anthropic
from anthropic.types import MessageParam, TextBlockParam
import pandas as pd
from IPython.display import display

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your Anthropic API key: ")
client = Anthropic(
    api_key=os.environ["ANTHROPIC_API_KEY"],
    timeout=90,
    max_retries=1,
)
for name, value in list(globals().items()):
    if name.startswith("RUN_") and not isinstance(value, bool):
        raise TypeError(f"{name} must be a Python boolean, not text")
recorded_messages = []

In [ ]:
available_models = list(client.models.list())
assert any(m.id == MODEL for m in available_models), "Choose an available model ID"
display(pd.DataFrame([{"id": m.id, "name": m.display_name} for m in available_models]))

## A basic request and response

Claude uses `client.messages.create()`, a required `max_tokens`, and a `messages` array. Initial instructions go in top-level `system`. The result contains typed `content` blocks and a `stop_reason`.

OpenAI Responses uses `client.responses.create()`, `instructions` and `input`. Its SDK provides `response.output_text`; raw output is an array of typed items. For Claude, select text blocks rather than assuming that `content[0]` is always text. These are native APIs, not interchangeable request bodies.

In [ ]:
response = client.messages.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    output_config={"effort": "low"},
    system="Explain concepts clearly for a beginner.",
    messages=[
        {"role": "user", "content": "Explain a model gateway in two short sentences."}
    ],
)
recorded_messages.append(("basic text", response))
assert response.stop_reason == "end_turn", f"Incomplete answer: {response.stop_reason}"
print("".join(block.text for block in response.content if block.type == "text"))
print("Response ID:", response.id, "Stop reason:", response.stop_reason)
print("Block types:", [b.type for b in response.content])

## Multi-turn conversation

The native Messages API needs prior conversation content in the next request. Preserve complete assistant content blocks, especially when using tools or thinking. There is no OpenAI-style `previous_response_id` on this interface. Claude Managed Agents is a separate stateful product.

Some newer Claude models also allow system messages later in a conversation, with placement rules. For the introductory model here, put initial instructions in top-level `system`. [Conversation patterns](https://platform.claude.com/docs/en/build-with-claude/working-with-messages)

In [ ]:
history: list[MessageParam] = [
    {"role": "user", "content": "Give me a one-sentence definition of a model gateway."}
]
first = client.messages.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    output_config={"effort": "low"},
    system="Be concise.",
    messages=history,
)
recorded_messages.append(("conversation turn 1", first))
history.append(
    {
        "role": "assistant",
        "content": first.content,
    }
)
history.append(
    {"role": "user", "content": "Now give me one practical reason to use it."}
)
second = client.messages.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    output_config={"effort": "low"},
    system="Be concise.",
    messages=history,
)
recorded_messages.append(("conversation turn 2", second))
assert second.stop_reason == "end_turn", f"Incomplete answer: {second.stop_reason}"
print("".join(block.text for block in second.content if block.type == "text"))

## Stream text and capture final usage

Both providers can stream an answer. The events and SDK helpers differ. Claude's helper below assembles the final message, which includes usage and the reason the turn stopped. [Streaming](https://platform.claude.com/docs/en/build-with-claude/streaming)

In [ ]:
with client.messages.stream(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    output_config={"effort": "low"},
    messages=[
        {"role": "user", "content": "Give two practical uses for a model gateway."}
    ],
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)
    final_message = stream.get_final_message()
print()
recorded_messages.append(("streaming", final_message))
assert final_message.stop_reason == "end_turn"

## Structured JSON

Use the native `output_config.format` schema. OpenAI Responses uses `text.format` for the same broad goal. Neither a JSON-shaped prompt nor a compatibility endpoint automatically provides the same schema guarantees. Validate the parsed values in your application. [Structured outputs](https://platform.claude.com/docs/en/build-with-claude/structured-outputs)

In [ ]:
review_schema = {
    "type": "object",
    "properties": {
        "sentiment": {"type": "string", "enum": ["positive", "negative", "neutral"]},
        "reason": {"type": "string"},
    },
    "required": ["sentiment", "reason"],
    "additionalProperties": False,
}
response = client.messages.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    messages=[
        {
            "role": "user",
            "content": "Classify: The mug looks lovely and delivery was fast.",
        }
    ],
    output_config={
        "effort": "low",
        "format": {"type": "json_schema", "schema": review_schema},
    },
)
recorded_messages.append(("structured JSON", response))
assert response.stop_reason == "end_turn", f"Incomplete answer: {response.stop_reason}"
review = json.loads(
    "".join(block.text for block in response.content if block.type == "text")
)
assert set(review) == {"sentiment", "reason"}
assert review["sentiment"].lower() in {"positive", "negative", "neutral"}
assert isinstance(review["reason"], str)
display(review)

## Token counting and usage

Use the provider's tokenizer rather than treating OpenAI's tiktoken count as Claude's exact count. Each model turn has its own input/output usage; cache reads, cache creation and hosted tools can have separate billing. Usage is not an invoice or a universal dollar-cost field.

[Token counting](https://platform.claude.com/docs/en/build-with-claude/token-counting) · [Pricing](https://platform.claude.com/docs/en/about-claude/pricing)

In [ ]:
count = client.messages.count_tokens(
    model=MODEL,
    system="Be concise.",
    messages=[{"role": "user", "content": "Explain a model gateway."}],
)
print("Estimated input tokens:", count.input_tokens)
display(
    pd.DataFrame(
        [
            {"purpose": purpose, "model": recorded.model, **recorded.usage.model_dump()}
            for purpose, recorded in recorded_messages
        ]
    )
)

## Optional: compare the same task with OpenAI

Enable `RUN_OPENAI_COMPARISON` to run the same small classification set through both native APIs. We use the same schema and acceptance criteria, with provider-specific adapters. This is a teaching evaluation, not proof of a universal winner. One run's latency is affected by network conditions and server load. Tokens use different tokenizers; compare actual billing separately.

Our application owns the common task and result format. Each adapter owns its provider's request shape.

In [ ]:
if RUN_OPENAI_COMPARISON:
    from openai import OpenAI

    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
    openai_client = OpenAI(timeout=90, max_retries=1)
    tests = [
        ("The mug is lovely and arrived early.", "positive"),
        ("The handle was broken and delivery was late.", "negative"),
        ("The package contains one blue mug.", "neutral"),
    ]
    results = []
    for text, expected in tests:
        for provider in ["Anthropic", "OpenAI"]:
            started = time.perf_counter()
            prompt = "Classify this review: " + text
            if provider == "Anthropic":
                r = client.messages.create(
                    model=MODEL,
                    max_tokens=MAX_TOKENS,
                    messages=[{"role": "user", "content": prompt}],
                    output_config={
                        "effort": "low",
                        "format": {"type": "json_schema", "schema": review_schema}
                    },
                )
                recorded_messages.append(("provider comparison", r))
                assert r.stop_reason == "end_turn", (
                    f"Incomplete answer: {r.stop_reason}"
                )
                parsed = json.loads(
                    "".join(block.text for block in r.content if block.type == "text")
                )
                input_tokens, output_tokens = (
                    r.usage.input_tokens,
                    r.usage.output_tokens,
                )
                selected_model = MODEL
            else:
                r = openai_client.responses.create(
                    model=OPENAI_MODEL,
                    input=prompt,
                    max_output_tokens=400,
                    reasoning={"effort": "none"},
                    text={
                        "format": {
                            "type": "json_schema",
                            "name": "review",
                            "strict": True,
                            "schema": review_schema,
                        }
                    },
                )
                if r.status != "completed":
                    raise RuntimeError(f"OpenAI response incomplete: {r.status}")
                parsed = json.loads(r.output_text)
                assert r.usage is not None, "OpenAI response did not include token usage"
                input_tokens, output_tokens = (
                    r.usage.input_tokens,
                    r.usage.output_tokens,
                )
                selected_model = OPENAI_MODEL
            results.append(
                {
                    "provider": provider,
                    "model": selected_model,
                    "review": text,
                    "label": parsed["sentiment"],
                    "passes": parsed["sentiment"].lower() == expected,
                    "seconds": round(time.perf_counter() - started, 2),
                    "input_tokens": input_tokens,
                    "output_tokens": output_tokens,
                }
            )
    display(pd.DataFrame(results))

## Optional: image and PDF inputs

Pass a public image URL directly to the SDK, or encode a local PDF as a document block. OpenAI uses different input item shapes. Enable one flag at a time and check the model's supported modalities. The PDF example needs a small local file; upload it through Colab's Files panel and set `PDF_PATH`.

[Vision](https://platform.claude.com/docs/en/build-with-claude/vision) · [PDF support](https://platform.claude.com/docs/en/build-with-claude/pdf-support)

In [ ]:
if RUN_VISION:
    response = client.messages.create(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        output_config={"effort": "low"},
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "image", "source": {"type": "url", "url": IMAGE_URL}},
                    {
                        "type": "text",
                        "text": "Explain the process in this diagram in one sentence.",
                    },
                ],
            }
        ],
    )
    recorded_messages.append(("vision", response))
    assert response.stop_reason == "end_turn", (
        f"Incomplete answer: {response.stop_reason}"
    )
    print("".join(block.text for block in response.content if block.type == "text"))

if RUN_PDF:
    if not PDF_PATH:
        print("PDF example skipped: set PDF_PATH to a small PDF to run it.")

if RUN_PDF and PDF_PATH:
    path = Path(PDF_PATH)
    if not path.is_file() or path.suffix.lower() != ".pdf":
        raise ValueError("Set PDF_PATH to a small PDF")
    response = client.messages.create(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        output_config={"effort": "low"},
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "document",
                        "source": {
                            "type": "base64",
                            "media_type": "application/pdf",
                            "data": base64.b64encode(path.read_bytes()).decode(),
                        },
                    },
                    {"type": "text", "text": "Summarise this PDF in three sentences."},
                ],
            }
        ],
    )
    recorded_messages.append(("PDF", response))
    assert response.stop_reason == "end_turn", (
        f"Incomplete answer: {response.stop_reason}"
    )
    print("".join(block.text for block in response.content if block.type == "text"))

## Optional: prompt caching

Caching reuses an eligible input prefix, not the entire final answer. Minimum lengths and supported cache options depend on the model. The example supplies a deliberately long teaching policy; do not add irrelevant text to production prompts just to obtain cache hits. Inspect actual cache-read/cache-creation usage instead of assuming that a second request was cached. [Prompt caching](https://platform.claude.com/docs/en/build-with-claude/prompt-caching)

In [ ]:
if RUN_CACHING:
    policy = "\n".join(
        f"Policy record {i}: UK shipping is £3; US shipping is £9; use the relevant country."
        for i in range(1200)
    )
    cached_system: list[TextBlockParam] = [
        {"type": "text", "text": policy, "cache_control": {"type": "ephemeral"}}
    ]
    for question in ["What is UK shipping?", "What is US shipping?"]:
        response = client.messages.create(
            model=MODEL,
            max_tokens=MAX_TOKENS,
            output_config={"effort": "low"},
            system=cached_system,
            messages=[{"role": "user", "content": question}],
        )
        recorded_messages.append(("prompt cache", response))
        assert response.stop_reason == "end_turn", (
            f"Incomplete answer: {response.stop_reason}"
        )
        print("".join(block.text for block in response.content if block.type == "text"))
        display(response.usage.model_dump())

## Optional: Anthropic-hosted search and code execution

These run on Anthropic's servers. Your custom Python functions run in your application. Hosted tools can add charges and may require account enablement or another supported model. A paused turn is not a finished answer; the examples stop explicitly if that happens.

[Web search](https://platform.claude.com/docs/en/agents-and-tools/tool-use/web-search-tool) · [Code execution](https://platform.claude.com/docs/en/agents-and-tools/tool-use/code-execution-tool)

In [ ]:
if RUN_SEARCH:
    response = client.messages.create(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        output_config={"effort": "low"},
        tools=[
            {
                "type": "web_search_20250305",
                "name": "web_search",
                "max_uses": 1,
                "allowed_domains": ["platform.claude.com"],
            }
        ],
        messages=[
            {
                "role": "user",
                "content": "Search for Claude's official Messages API guide "
                "and cite the source.",
            }
        ],
    )
    recorded_messages.append(("hosted search", response))
    assert response.stop_reason == "end_turn", (
        f"Incomplete answer: {response.stop_reason}"
    )
    print("".join(block.text for block in response.content if block.type == "text"))
    for block in response.content:
        if block.type == "text":
            display([citation.model_dump() for citation in (block.citations or [])])

if RUN_CODE_EXECUTION:
    response = client.messages.create(
        model=SERVER_TOOL_MODEL,
        max_tokens=MAX_TOKENS,
        output_config={"effort": "low"},
        tools=[{"type": "code_execution_20260521", "name": "code_execution"}],
        messages=[
            {
                "role": "user",
                "content": "Use code execution to run Python and verify the sum "
                "of integers from 1 to 100. Give the result.",
            }
        ],
    )
    recorded_messages.append(("hosted Python", response))
    assert response.stop_reason == "end_turn", (
        f"Incomplete answer: {response.stop_reason}"
    )
    print("".join(block.text for block in response.content if block.type == "text"))
    print("Evidence block types:", [b.type for b in response.content])
    assert any(b.type == "server_tool_use" for b in response.content), (
        "No server tool execution evidence"
    )

## Optional: batch processing

Submit work that does not need an immediate answer. Batch submission is not completion. Retrieve status/results later using the saved ID; this notebook does not wait for a long-running job. Enable `RUN_BATCH` only when you intend to create a paid batch. [Batches](https://platform.claude.com/docs/en/build-with-claude/batch-processing)

In [ ]:
if RUN_BATCH:
    batch = client.messages.batches.create(
        requests=[
            {
                "custom_id": "gateway-summary",
                "params": {
                    "model": MODEL,
                    "max_tokens": MAX_TOKENS,
                    "output_config": {"effort": "low"},
                    "messages": [
                        {
                            "role": "user",
                            "content": "Define a model gateway in one sentence.",
                        }
                    ],
                },
            }
        ]
    )
    BATCH_ID = batch.id
    print("Save this batch ID:", BATCH_ID)
if BATCH_ID:
    status = client.messages.batches.retrieve(BATCH_ID)
    print("Batch state:", status.processing_status)
    if status.processing_status == "ended":
        for result in client.messages.batches.results(BATCH_ID):
            display(result.model_dump())

## Similar goals, different contracts

| Concept | OpenAI Responses | Claude Messages |
|---|---|---|
| Client call | `responses.create()` | `messages.create()` |
| Initial guidance | `instructions` or appropriate message items | Top-level `system` |
| User input | `input` string or typed items | `messages` with content blocks |
| Output | Typed `output` items; SDK `output_text` helper | Typed `content` blocks; select text blocks |
| Continuation | Manual items, `previous_response_id`, or Conversations API | Send prior messages/content on Messages API |
| JSON schema | `text.format` | `output_config.format` |
| Client tools | `function_call` / `function_call_output` | `tool_use` / `tool_result` |
| Usage | Provider-specific response usage | Provider-specific message usage |

**Embeddings:** Anthropic's guide recommends assessing an external embeddings provider rather than calling a Claude-native embeddings model. This is a concrete reason to combine providers in one application. [Embeddings guide](https://platform.claude.com/docs/en/build-with-claude/embeddings)

## Summary

- Claude's native SDK uses Messages and typed content blocks.
- Text, conversation, streaming, JSON and tools are familiar concepts with different API shapes.
- Preserve content blocks and handle stop reasons; a truncated or paused response is not a successful final answer.
- Reuse your task, schemas and acceptance criteria across providers, with provider-specific adapters.
- Choose the model, tools and context using evaluations of your actual task. There is no permanent universal winner.

Next: [Claude prompting and tools](claude_prompting_and_tools.ipynb).